## Libraries

In [ ]:
import math
import pandas as pd
import torch
import os
from pprint import pprint
import subprocess
from urllib.request import urlopen
from io import BytesIO
from zipfile import ZipFile

## Download Greek data

In [2]:
greek_data_url = 'https://github.com/iit-Demokritos/FNS2023_data/raw/refs/heads/main/Greek.zip'
http_response = urlopen(greek_data_url)
zipfile = ZipFile(BytesIO(http_response.read()))
zipfile.extractall(path='.')

In [3]:
ANNUAL_REPORTS_DIR = 'Greek/training/annual_reports/'
GOLD_SUMMARIES_DIR = 'Greek/training/gold_summaries/'
CANDIDATE_SUMMARIES_DIR = 'Greek/training/candidate_summaries/'

## Sample source docs

In [4]:
source_docs = [file[:-4] for file in os.listdir(ANNUAL_REPORTS_DIR)[:10]]

## Candidate summaries

### Abstractive summaries with a summarization model

In [6]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("kriton/greek-text-summarization")
model = AutoModelForSeq2SeqLM.from_pretrained("kriton/greek-text-summarization")

In [7]:
def get_token_length(filepath):
    with open(filepath, mode='r', encoding='utf-8') as file:
        text = file.read()
    return len(tokenizer(text, truncation=False, padding=False)['input_ids'])

In [8]:
source_token_lengths = [
    get_token_length(ANNUAL_REPORTS_DIR+txtfile)
    for txtfile in os.listdir(ANNUAL_REPORTS_DIR)
]

goldsum_token_lengths = [
    get_token_length(GOLD_SUMMARIES_DIR+txtfile)
    for txtfile in os.listdir(GOLD_SUMMARIES_DIR)    
]

In [9]:
max_input_token_length = max(source_token_lengths)
min_output_token_length = min(goldsum_token_lengths)
max_output_token_length = max(goldsum_token_lengths)

In [10]:
print(max_input_token_length)
print(min_output_token_length)
print(max_output_token_length)

582189
1
159722


In [ ]:
from transformers import pipeline

summarizer = pipeline("summarization", model="kriton/greek-text-summarization")

def genarate_summary(article):
    inputs = tokenizer(
        'summarize: ' + article, 
        return_tensors="pt", 
        max_length=max_input_token_length, 
        truncation=True,
        padding="max_length",
    )

    outputs = model.generate(
        inputs["input_ids"], 
        max_length=max_output_token_length, 
        min_length=min_output_token_length, 
        length_penalty=3.0,
        num_beams=8, 
        early_stopping=True,
        repetition_penalty=3.0,
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

Device set to use cuda:0


In [24]:
for txtfile in os.listdir(ANNUAL_REPORTS_DIR)[:1]:
    with open(ANNUAL_REPORTS_DIR+txtfile, mode='r', encoding='utf-8') as file:
        source = file.read()
        print(genarate_summary(article=source))

Σημαντικά γεγονότα της χρήσης για το έτος που έληξε την 31η Δεκεμβρίου 2019 (Σύμφωνα με το Ν.3556/07) Ο Αναπληρωτής Διευθύνων Σύμβουλος Μάρκος Μπιτσάκος αναφέρει: Ετήσια Οικονομική Έκθεση της 31 Δεκεμβρίου 2019 (Τα ποσά παρουσιάζονται σε χιλιάδες Ευρώ εκτός εάν δηλώνεται διαφορετικά) δ


### Reduced gold summaries as candidate summaries

In [37]:
import random
from random import shuffle

In [45]:
with open(GOLD_SUMMARIES_DIR+'1_1.txt', mode='r', encoding='utf-8') as file:
    gold_summary = file.read()
    print(gold_summary)

Τα μέλη του Διοικητικού Συμβουλίου, κ.κ. Θεόδωρος Φέσσας, Πρόεδρος, Απόστολος Γεωργαντζής, Διευθύνων
Σύμβουλος και Μάρκος Μπιτσάκος, Αναπληρωτής Διευθύνων Σύμβουλος , υπό την ως άνω ιδιότητά τους, δηλώνουν
ότι, εξ όσων γνωρίζουν :


Οι συνημμένες ετήσιες εταιρικές και ενοποιημένες Χρηματοοικονομικές Καταστάσεις της Quest Συμμετοχών
A.E. (η Εταιρεία), οι οποίες καταρτίσθηκαν για τη χρήση 1/1-31/12/2019, σύμφωνα με τα Διεθνή Πρότυπα
Χρηματοοικονομικής Αναφοράς, απεικονίζουν κατά τρόπο αληθή τα στοιχεία του ενεργητικού και του
παθητικού, την καθαρή θέση και τα αποτελέσματα χρήσης της Εταιρείας, καθώς και των εταιρειών που
περιλαμβάνονται στην ενοποίηση εκλαμβανομένων ως σύνολο (ο Όμιλος).



Η συνημμένη έκθεση του Διοικητικού Συμβουλίου απεικονίζει κατά τρόπον αληθή την εξέλιξη, τις επιδόσεις
και τη θέση της Εταιρείας, καθώς και του Ομίλου, συμπεριλαμβανομένης της περιγραφής των κυριότερων
κινδύνων και αβεβαιοτήτων που αντιμετωπίζουν.

Καλλιθέα, 7 Απριλίου 2020

Ο Πρόεδρος

Θεόδωρος Φέσ

#### Shuffle

In [255]:
def shuffle_words(target):
    with open(GOLD_SUMMARIES_DIR+f'{target}_1.txt', mode='r', encoding='utf-8') as file:
        gold_summary = file.read().split()
        shuffle(gold_summary)
        return ' '.join(list)

pprint(shuffle_words(target='1'))

('όσων A.E. καθαρή εξέλιξη, Θεόδωρος Ο Διευθύνων Αναφοράς, τις Απριλίου Τα '
 'Σύμβουλος τους, (ο Οι δηλώνουν έκθεση Μπιτσάκος, απεικονίζει Πρόεδρος και '
 'την Χρηματοοικονομικές της στην Πρότυπα μέλη και ενοποιημένες Φέσσας και '
 'συνημμένες \uf0b7 και ιδιότητά Φέσσας, γνωρίζουν Ο κατά Όμιλος). '
 'περιλαμβάνονται ενοποίηση τρόπον στοιχεία κυριότερων τη του σύμφωνα κ.κ. '
 'Θεόδωρος ως Μάρκος Η Καλλιθέα, επιδόσεις και Διευθύνων άνω θέση '
 'εκλαμβανομένων Σύμβουλος ενεργητικού Γεωργαντζής Συμβουλίου οποίες για της '
 'αβεβαιοτήτων την ότι, Εταιρείας, Εταιρεία), τα αληθή την της Ομίλου, χρήση '
 'σύνολο τα της και και καθώς συμπεριλαμβανομένης Quest Χρηματοοικονομικής ως '
 '2020 του εξ απεικονίζουν καθώς συνημμένη (η που , Καταστάσεις χρήσης '
 'Εταιρείας, Συμμετοχών κινδύνων υπό Διευθύνων Συμβουλίου, Διοικητικού των '
 'εταιρειών κατά του \uf0b7 Απόστολος Γεωργαντζής, αντιμετωπίζουν. 7 '
 '1/1-31/12/2019, αληθή τη Πρόεδρος, τα των του και ετήσιες αποτελέσματα : '
 'περιγραφής Διοικ

#### Randomly swap words

In [39]:
import spacy

nlp = spacy.load('el_core_news_sm')

In [47]:
with open(GOLD_SUMMARIES_DIR+'1_1.txt', mode='r', encoding='utf-8') as file:
    gold_summary = file.read()
    doc = nlp(gold_summary)
    print([(token.text, token.pos_) for token in doc])

[('Τα', 'DET'), ('μέλη', 'NOUN'), ('του', 'DET'), ('Διοικητικού', 'ADJ'), ('Συμβουλίου', 'NOUN'), (',', 'PUNCT'), ('κ.κ.', 'VERB'), ('Θεόδωρος', 'NUM'), ('Φέσσας', 'NOUN'), (',', 'PUNCT'), ('Πρόεδρος', 'NOUN'), (',', 'PUNCT'), ('Απόστολος', 'PROPN'), ('Γεωργαντζής', 'X'), (',', 'PUNCT'), ('Διευθύνων', 'PROPN'), ('\n', 'SPACE'), ('Σύμβουλος', 'PROPN'), ('και', 'CCONJ'), ('Μάρκος', 'X'), ('Μπιτσάκος', 'PROPN'), (',', 'PUNCT'), ('Αναπληρωτής', 'NOUN'), ('Διευθύνων', 'NOUN'), ('Σύμβουλος', 'PROPN'), (',', 'PUNCT'), ('υπό', 'ADP'), ('την', 'DET'), ('ως', 'ADV'), ('άνω', 'ADV'), ('ιδιότητά', 'NOUN'), ('τους', 'PRON'), (',', 'PUNCT'), ('δηλώνουν', 'VERB'), ('\n', 'SPACE'), ('ότι', 'SCONJ'), (',', 'PUNCT'), ('εξ', 'ADP'), ('όσων', 'PRON'), ('γνωρίζουν', 'VERB'), (':', 'PUNCT'), ('\n', 'SPACE'), ('\uf0b7', 'NOUN'), ('\n\n', 'SPACE'), ('Οι', 'DET'), ('συνημμένες', 'ADJ'), ('ετήσιες', 'ADJ'), ('εταιρικές', 'ADJ'), ('και', 'CCONJ'), ('ενοποιημένες', 'VERB'), ('Χρηματοοικονομικές', 'ADJ'), ('Καταστ

In [65]:
from pprint import pprint

In [256]:
def swap_words(target):
    with open(GOLD_SUMMARIES_DIR+f'{target}_1.txt', mode='r', encoding='utf-8') as file:
        gold_summary = file.read().split()
        for swaps in range(10):
            idx = range(len(gold_summary))
            i1, i2 = random.sample(idx, 2)
            gold_summary[i1], gold_summary[i2] = gold_summary[i2], gold_summary[i1]
        return ' '.join(gold_summary)

pprint(swap_words(target='1'))

('Τα μέλη του Διοικητικού Συμβουλίου, κ.κ. Θεόδωρος Φέσσας, 7 Απόστολος '
 'Γεωργαντζής, Διευθύνων Σύμβουλος και Μάρκος Μπιτσάκος, Αναπληρωτής Διευθύνων '
 'Σύμβουλος επιδόσεις υπό την ως άνω ιδιότητά τους, Απόστολος ότι, εξ όσων '
 'γνωρίζουν : \uf0b7 Οι συνημμένες ετήσιες εταιρικές και ενοποιημένες '
 'Χρηματοοικονομικές Καταστάσεις της Quest Συμμετοχών A.E. (η Εταιρεία), οι '
 'οποίες της για τη χρήση 1/1-31/12/2019, σύμφωνα με τα Διεθνή Πρότυπα '
 'Χρηματοοικονομικής του του κατά τρόπο της τα στοιχεία του ενεργητικού και '
 'απεικονίζουν παθητικού, την καθαρή καθώς και τα αποτελέσματα χρήσης της '
 'Εταιρείας, θέση και των εταιρειών που Εταιρείας, στην ενοποίηση '
 'εκλαμβανομένων ως σύνολο (ο Όμιλος). \uf0b7 Η συνημμένη έκθεση του '
 'Διοικητικού Συμβουλίου απεικονίζει θέση τρόπον αληθή την εξέλιξη, τις , και '
 'τη κατά αληθή περιλαμβάνονται καθώς και Αναφοράς, Ομίλου, '
 'συμπεριλαμβανομένης καταρτίσθηκαν περιγραφής των κυριότερων κινδύνων και '
 'αβεβαιοτήτων που αντιμετωπίζουν

#### Randomly delete words

In [257]:
def delete_words(target):
    with open(GOLD_SUMMARIES_DIR+f'{target}_1.txt', mode='r', encoding='utf-8') as file:
        gold_summary = file.read()
        gold_summary_list = gold_summary.split()
        for removals in range(20):
            random_word = random.choice(gold_summary_list)
            gold_summary_list.remove(random_word)
        return ' '.join(gold_summary_list)

pprint(delete_words(target='1'))

('Τα Διοικητικού Συμβουλίου, κ.κ. Θεόδωρος Φέσσας, Γεωργαντζής, Σύμβουλος και '
 'Μάρκος Μπιτσάκος, Αναπληρωτής Διευθύνων Σύμβουλος , υπό ως άνω ιδιότητά '
 'τους, δηλώνουν ότι, εξ όσων γνωρίζουν Οι συνημμένες εταιρικές και '
 'ενοποιημένες Καταστάσεις Quest Συμμετοχών A.E. (η Εταιρεία), οποίες '
 'καταρτίσθηκαν για χρήση 1/1-31/12/2019, σύμφωνα με τα Διεθνή Πρότυπα '
 'Χρηματοοικονομικής Αναφοράς, απεικονίζουν κατά τρόπο αληθή τα στοιχεία του '
 'ενεργητικού και του παθητικού, την καθαρή θέση και τα αποτελέσματα χρήσης '
 'της Εταιρείας, καθώς και εταιρειών περιλαμβάνονται στην ενοποίηση '
 'εκλαμβανομένων ως σύνολο (ο Όμιλος). \uf0b7 Η συνημμένη έκθεση του '
 'Διοικητικού Συμβουλίου απεικονίζει κατά αληθή την εξέλιξη, τις επιδόσεις και '
 'τη θέση της Εταιρείας, καθώς και του συμπεριλαμβανομένης της περιγραφής των '
 'κινδύνων και αβεβαιοτήτων που αντιμετωπίζουν. Απριλίου 2020 Ο Πρόεδρος '
 'Θεόδωρος Φέσσας Ο Διευθύνων Σύμβουλος Απόστολος Γεωργαντζής')


#### Randomly remove sentences

In [259]:
# Splitting with spacy didnt work

def remove_sentence(target):
    with open(GOLD_SUMMARIES_DIR+f'{target}_1.txt', mode='r', encoding='utf-8') as file:
        gold_summary = file.read()
        sentences = gold_summary.split('.')
        sent_remove_len = 0
        while sent_remove_len < 2:
            sent_remove = random.choice(sentences)
            sent_remove_len = len(sent_remove)
        new_text = [sent for sent in sentences if sent != sent_remove]
        return ''.join(new_text)

pprint(remove_sentence(target='1'))

('Τα μέλη του Διοικητικού Συμβουλίου, κκ Θεόδωρος Φέσσας, Πρόεδρος, Απόστολος '
 'Γεωργαντζής, Διευθύνων\n'
 'Σύμβουλος και Μάρκος Μπιτσάκος, Αναπληρωτής Διευθύνων Σύμβουλος , υπό την ως '
 'άνω ιδιότητά τους, δηλώνουν\n'
 'ότι, εξ όσων γνωρίζουν :\n'
 '\uf0b7\n'
 '\n'
 'Οι συνημμένες ετήσιες εταιρικές και ενοποιημένες Χρηματοοικονομικές '
 'Καταστάσεις της Quest Συμμετοχών\n'
 'AE (η Εταιρεία), οι οποίες καταρτίσθηκαν για τη χρήση 1/1-31/12/2019, '
 'σύμφωνα με τα Διεθνή Πρότυπα\n'
 'Χρηματοοικονομικής Αναφοράς, απεικονίζουν κατά τρόπο αληθή τα στοιχεία του '
 'ενεργητικού και του\n'
 'παθητικού, την καθαρή θέση και τα αποτελέσματα χρήσης της Εταιρείας, καθώς '
 'και των εταιρειών που\n'
 'περιλαμβάνονται στην ενοποίηση εκλαμβανομένων ως σύνολο (ο Όμιλος)\n'
 '\n'
 '\uf0b7\n'
 '\n'
 'Η συνημμένη έκθεση του Διοικητικού Συμβουλίου απεικονίζει κατά τρόπον αληθή '
 'την εξέλιξη, τις επιδόσεις\n'
 'και τη θέση της Εταιρείας, καθώς και του Ομίλου, συμπεριλαμβανομένης της '
 'περιγραφής των 

#### Insert sentences from other summaries

In [261]:
# Splitting with spacy didnt work

def insert_sentence(target):
    with open(GOLD_SUMMARIES_DIR+f'{target}_1.txt', mode='r', encoding='utf-8') as file1:
        gold_summary = file1.read()
        sentences = gold_summary.split('.')

        # Choose another summary
        remaining_summaries = [doc for doc in source_docs if doc != '1']
        rand_summary = random.choice(remaining_summaries)

        # Choose a sentence from the randomly picked summary to insert to the original one
        with open(GOLD_SUMMARIES_DIR+f'{rand_summary}_1.txt', mode='r', encoding='utf-8') as file2:
            rand_gold_summary = file2.read()
            rand_sentences = rand_gold_summary.split('.')
            sent_insert_len = 0
            while sent_insert_len < 2:
                sent_insert = random.choice(rand_sentences)
                sent_insert_len = len(sent_insert)

        new_text = [sent for sent in sentences]
        
        return ''.join(new_text) + '' + sent_insert

pprint(insert_sentence(target='1'))

('Τα μέλη του Διοικητικού Συμβουλίου, κκ Θεόδωρος Φέσσας, Πρόεδρος, Απόστολος '
 'Γεωργαντζής, Διευθύνων\n'
 'Σύμβουλος και Μάρκος Μπιτσάκος, Αναπληρωτής Διευθύνων Σύμβουλος , υπό την ως '
 'άνω ιδιότητά τους, δηλώνουν\n'
 'ότι, εξ όσων γνωρίζουν :\n'
 '\uf0b7\n'
 '\n'
 'Οι συνημμένες ετήσιες εταιρικές και ενοποιημένες Χρηματοοικονομικές '
 'Καταστάσεις της Quest Συμμετοχών\n'
 'AE (η Εταιρεία), οι οποίες καταρτίσθηκαν για τη χρήση 1/1-31/12/2019, '
 'σύμφωνα με τα Διεθνή Πρότυπα\n'
 'Χρηματοοικονομικής Αναφοράς, απεικονίζουν κατά τρόπο αληθή τα στοιχεία του '
 'ενεργητικού και του\n'
 'παθητικού, την καθαρή θέση και τα αποτελέσματα χρήσης της Εταιρείας, καθώς '
 'και των εταιρειών που\n'
 'περιλαμβάνονται στην ενοποίηση εκλαμβανομένων ως σύνολο (ο Όμιλος)\n'
 '\n'
 '\uf0b7\n'
 '\n'
 'Η συνημμένη έκθεση του Διοικητικού Συμβουλίου απεικονίζει κατά τρόπον αληθή '
 'την εξέλιξη, τις επιδόσεις\n'
 'και τη θέση της Εταιρείας, καθώς και του Ομίλου, συμπεριλαμβανομένης της '
 'περιγραφής των 

#### Repeat the same sentence

In [262]:
# Splitting with spacy didnt work

def repeat_sentence(target):
    with open(GOLD_SUMMARIES_DIR+f'{target}_1.txt', mode='r', encoding='utf-8') as file1:
        gold_summary = file1.read()
        sentences = gold_summary.split('.')
        sent_repeat_len = 0
        while sent_repeat_len < 2:
            sent_repeat = random.choice(sentences)
            sent_repeat_len = len(sent_repeat)

        repeated_sent = [sent_repeat for sent_repeat in range(8)]
        new_text = [sent for sent in sentences]
        
        return ''.join(new_text) + '' + ''.join(repeated_sent)

pprint(repeat_sentence(target='1'))

('Τα μέλη του Διοικητικού Συμβουλίου, κκ Θεόδωρος Φέσσας, Πρόεδρος, Απόστολος '
 'Γεωργαντζής, Διευθύνων\n'
 'Σύμβουλος και Μάρκος Μπιτσάκος, Αναπληρωτής Διευθύνων Σύμβουλος , υπό την ως '
 'άνω ιδιότητά τους, δηλώνουν\n'
 'ότι, εξ όσων γνωρίζουν :\n'
 '\uf0b7\n'
 '\n'
 'Οι συνημμένες ετήσιες εταιρικές και ενοποιημένες Χρηματοοικονομικές '
 'Καταστάσεις της Quest Συμμετοχών\n'
 'AE (η Εταιρεία), οι οποίες καταρτίσθηκαν για τη χρήση 1/1-31/12/2019, '
 'σύμφωνα με τα Διεθνή Πρότυπα\n'
 'Χρηματοοικονομικής Αναφοράς, απεικονίζουν κατά τρόπο αληθή τα στοιχεία του '
 'ενεργητικού και του\n'
 'παθητικού, την καθαρή θέση και τα αποτελέσματα χρήσης της Εταιρείας, καθώς '
 'και των εταιρειών που\n'
 'περιλαμβάνονται στην ενοποίηση εκλαμβανομένων ως σύνολο (ο Όμιλος)\n'
 '\n'
 '\uf0b7\n'
 '\n'
 'Η συνημμένη έκθεση του Διοικητικού Συμβουλίου απεικονίζει κατά τρόπον αληθή '
 'την εξέλιξη, τις επιδόσεις\n'
 'και τη θέση της Εταιρείας, καθώς και του Ομίλου, συμπεριλαμβανομένης της '
 'περιγραφής των 

### Candidate summaries generation

In [274]:
def generate_noisy_summaries(doc, path):
    with open(path+f'{doc}_shuffled_words.txt', mode='w', encoding='utf-8') as file:
        file.write(shuffle_words(target=doc))
    with open(path+f'{doc}_deleted_words.txt', mode='w', encoding='utf-8') as file:
        file.write(delete_words(target=doc))
    with open(path+f'{doc}_removed_sentence.txt', mode='w', encoding='utf-8') as file:
        file.write(remove_sentence(target=doc))
    with open(path+f'{doc}_inserted_sentence.txt', mode='w', encoding='utf-8') as file:
        file.write(insert_sentence(target=doc))
    with open(path+f'{doc}_repeated_sentence.txt', mode='w', encoding='utf-8') as file:
        file.write(repeat_sentence(target=doc))

for doc in source_docs:
    generate_noisy_summaries(doc=doc, path=CANDIDATE_SUMMARIES_DIR)

## Scoring

In [94]:
from rouge_score.rouge_scorer import RougeScorer
rouge1 = RougeScorer(['rouge1'], use_stemmer=True)
rouge2 = RougeScorer(['rouge2'], use_stemmer=True)

In [65]:
from transformers import BertTokenizer, BertForMaskedLM, BertModel
from bert_score import BERTScorer
bert_scorer = BERTScorer(lang='Greek') # bert-base-greek-uncased-v1 results in a key error

c:\Users\Stefania\Documents\Projects\thesis\.venv\Lib\site-packages\huggingface_hub\file_download.py:140: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Stefania\.cache\huggingface\hub\models--bert-base-multilingual-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [ ]:
import evaluate
bleurt = evaluate.load('bleurt', module_type='metric')

In [91]:
def rouge1_score(target, prediction):
    rouge1 = RougeScorer(['rouge1'], use_stemmer=True)
    return rouge1.score(target=target, prediction=prediction)['rouge1'][2]

def rouge2_score(target, prediction):
    rouge2 = RougeScorer(['rouge2'], use_stemmer=True)
    return rouge2.score(target=target, prediction=prediction)['rouge2'][2]

In [99]:
class NPowER:
    def __init__(self, min_n=3, max_n=3, dwin=3, min_score=0.0, max_score=1.0):
        self.min_n = min_n
        self.max_n = max_n
        self.dwin = dwin
        self.min_score = min_score
        self.max_score = max_score

    def score(self, target, prediction):
        result =  subprocess.check_output([
            "java",
            "-jar",
            "NPowERV1/V1/NPowER.jar",
            f"-peer={target}",
            f"-models={prediction}",
            f"[-minN={self.min_n}]",
            f"[-maxN={self.max_n}]",
            f"[-dwin={self.dwin}]",
            f"[-minScore={self.min_score}]",
            f"[-maxScore={self.max_score}]",
            "-allScores",
            "[-noSelfModel]"
            ], text=True)
        return result

npower = NPowER()

In [34]:
metrics = {
    'N-gram': {
        'Rouge1': rouge1_score,
        'Rouge2': rouge2_score,
    },
    'Graph': {
        'AutoSummENG': '',
    },
    'Probabilistic': {
        'SummTriver': '',
    },
    'Meta': {
        'NPowER': npower.score,
        'FRESA': '',
        'BRUGEscore': '',
    },
    'Embeddings': {
        'BERTScore': '',
        'BARTscore': '',
        'Bleurt': ''
    },
    'Transformers': {
        'GPTscore': '',
        'G-Eval': ''
    },
}

In [ ]:
for metric_type, metric_dict in metrics.items():
    for metric_name, metric_func in metric_dict.items():
        print(metric_func)
        score = metric_func(target='fff', prediction='fff')
        print(f'{metric_name}: {score}')

In [101]:
npower.score(target='Greek/training/gold_summaries/1_1.txt', prediction='Greek/training/gold_summaries/1_1.txt').strip().split()

['1.000000', '1.000000', '0.888240']

In [168]:
source_docs.copy()

['1', '10', '100', '101', '102', '103', '105', '106', '107', '108']

In [ ]:
data = {
    'source_doc': [],
    'type': [],
    'method': [],
    'variant': [],
    'score':[],
}

def append_score(data, source_file, type, method, candidate_variant, result):
    data['source_doc'].append(source_file)  
    data['type'].append(type)
    data['method'].append(method)
    data['variant'].append(candidate_variant)
    data['score'].append(result)


for source_file in source_docs:
    source_path = GOLD_SUMMARIES_DIR+source_file+'_1.txt'
    with open(source_path, mode='r', encoding='utf-8') as gold_f:
        gold_summary = gold_f.read()

    candidate_summaries = [doc for doc in os.listdir(CANDIDATE_SUMMARIES_DIR) if doc.startswith(f'{source_file}_' )]
    candidate_summaries.insert(0, source_file)

    for candidate_file in candidate_summaries:
        if candidate_file.isdigit():
            candidate_path = GOLD_SUMMARIES_DIR+candidate_file+'_1.txt'
            candidate_variant = 'source'
        else:
            candidate_path = CANDIDATE_SUMMARIES_DIR+candidate_file
            candidate_variant = candidate_file.removeprefix(f'{source_file}_').removesuffix('.txt')

        with open(candidate_path, mode='r', encoding='utf-8') as cand_f:
            candidate_summary = cand_f.read()

            # ----------N-GRAM    
            # Rouge 1
            rouge1_result = rouge1.score(target=gold_summary, prediction=candidate_summary)['rouge1'][2] #fmeasure
            append_score(source_file=source_file, type='N-gram', method='Rouge1', candidate_variant=candidate_variant, result=rouge1_result)

            # Rouge 2
            rouge2_result = rouge2.score(target=gold_summary, prediction=candidate_summary)['rouge2'][2] #fmeasure
            append_score(source_file=source_file, type='N-gram', method='Rouge2', candidate_variant=candidate_variant, result=rouge2_result)


            # ---------GRAPH
            autosummeng_score, memog_score, npower_score = npower.score(target=source_path, prediction=candidate_path).strip().split()

            # AutoSummENG
            append_score(source_file=source_file, type='Graph-based', method='AutoSummENG', candidate_variant=candidate_variant, result=autosummeng_score)

            # MeMoG
            append_score(source_file=source_file, type='Graph-based', method='MeMoG', candidate_variant=candidate_variant, result=memog_score)


            # ---------PROBABILISTIC
            # SummTriver


            # ---------META
            # NPowER - computed in graph methods
            append_score(source_file=source_file, type='Graph-based', method='NPowER', candidate_variant=candidate_variant, result=npower_score)

            # FRESA

            # BRUGEscore


            # ---------EMBEDDINGS-BASED
            # BERTScore
            P, R, F1 = bert_scorer.score([candidate_summary], [gold_summary])
            append_score(source_file=source_file, type='Embeddings-based', method='BERTScore', candidate_variant=candidate_variant, result=float(F1))

            # BARTscore

            # Bleurt


            # ---------TRANSFORMER-BASED
            # GPTscore

            # G-Eval

            # Extract-then-Evaluate

In [138]:
source_docs

['1', '10', '100', '101', '102', '103', '105', '106', '107', '108']

In [167]:
df = pd.DataFrame.from_dict(data, orient='index').transpose()
df.head(30)

,source_doc,type,method,variant,score
0,1,N-gram,Rouge1,source,1.0
1,1,N-gram,Rouge2,source,1.0
2,1,Graph-based,AutoSummENG,source,1.000000
3,1,Graph-based,MeMoG,source,1.000000
4,1,Graph-based,NPowER,source,0.888240
5,1,Embeddings-based,BERTScore,source,1.0
6,1,N-gram,Rouge1,deleted_words,1.0
7,1,N-gram,Rouge2,deleted_words,1.0
8,1,Graph-based,AutoSummENG,deleted_words,0.771729
9,1,Graph-based,MeMoG,deleted_words,0.771729
